# 01 · Préparation des ligands
Morphine et atropine (L-hyoscyamine) : SMILES isomériques (PubChem), stéréochimie, descripteurs, conformères 3D.

Ce notebook tourne dans **Google Colab** (cellule 1) ou en local avec l'environnement `drug-design`.

In [ ]:
# Cellule d'installation (Colab uniquement ; en local, utiliser environment.yml)
import os, sys
if 'google.colab' in sys.modules:
    !pip install -q rdkit
    if not os.path.exists('drug-design-morphine-paclitaxel'):
        !git clone -q https://github.com/AymaneAmri-MD/drug-design-morphine-paclitaxel.git
    %cd drug-design-morphine-paclitaxel/notebooks
os.makedirs('../results', exist_ok=True)
sys.path.append('../src')

In [ ]:
import requests
import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem import Draw
from ligand_prep import load_mol, describe, embed_conformers, write_best_conformer

print('RDKit', rdBase.rdkitVersion)

## Définition des ligands
Les SMILES isomériques (avec stéréochimie) sont récupérés directement depuis PubChem : par CID pour la morphine (5288826), par nom pour l'atropine.

On utilise le nom **hyoscyamine** (énantiomère L, actif) plutôt qu'« atropine », qui désigne le mélange racémique. Le premier résultat renvoyé par PubChem est utilisé : vérifie qu'il contient bien des `@`.

In [ ]:
def pubchem_smiles(ident):
    """SMILES isomérique depuis PubChem, par CID (int) ou par nom (str)."""
    kind = 'cid' if isinstance(ident, int) else 'name'
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{kind}/{ident}/property/IsomericSMILES/TXT"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.text.strip().splitlines()[0]

IDENTS = {'morphine': 5288826, 'atropine (L-hyoscyamine)': 'hyoscyamine'}
ligands = {name: pubchem_smiles(ident) for name, ident in IDENTS.items()}
for name, smi in ligands.items():
    print(name, '->', smi)
    assert '@' in smi, f'{name} : pas de stéréochimie dans le SMILES'

In [ ]:
mols = {k: load_mol(v) for k, v in ligands.items()}
Draw.MolsToGridImage(list(mols.values()), legends=list(mols.keys()), subImgSize=(400, 350))

## Descripteurs et contrôle de la stéréochimie
Valeurs attendues : **5 centres chiraux** pour la morphine, **4** pour la L-hyoscyamine (les 3 centres du noyau tropane, dont le C3 dit pseudo-asymétrique que RDKit note en minuscule, et le carbone de l'acide tropique).

Les deux molécules respectent la règle des 5 de Lipinski.

In [ ]:
EXPECTED_CHIRAL = {'morphine': 5, 'atropine (L-hyoscyamine)': 4}

table = pd.DataFrame({k: describe(m) for k, m in mols.items()}).T
display(table)

for name, expected in EXPECTED_CHIRAL.items():
    found = table.loc[name, 'centres_chiraux']
    status = 'OK' if found == expected else 'À VÉRIFIER'
    print(f'{name}: {found} centres chiraux (attendu {expected}) -> {status}')

## Conformères 3D
50 conformères par ligand, optimisation MMFF, seed fixé (42) pour la reproductibilité.
Les énergies MMFF ne se comparent qu'entre conformères d'une même molécule, pas entre molécules.

In [ ]:
for name, mol in mols.items():
    safe = name.split()[0]
    molh, ids, energies = embed_conformers(mol, n_confs=50)
    best = write_best_conformer(molh, energies, f'../results/{safe}_best.mol')
    print(f'{name}: {len(ids)} conformères, meilleur = {best}, E = {min(energies):.1f} kcal/mol')

## Étape suivante
`02_docking.ipynb` : re-docking du ligand co-cristallisé, puis docking morphine → récepteur µ et atropine → récepteur muscarinique M2.